# BAM to UMI Count Matrix

Streams a coordinate-sorted, STAR-aligned BAM file and produces a **cells × genes AnnData count matrix**.

- Cell barcodes and UMIs are parsed from the read name
- Gene assignment uses a GTF annotation via pyranges interval overlap
- UMI deduplication is done per (cell, gene) pair
- Output: `.h5ad` (compatible with `gene_expression_analysis.ipynb`) + `.csv`

**Workflow:**
1. Edit the CONFIG cell (Cell 1)
2. Run Cells 1–4b to validate inputs
3. Run Cells 5–12 to process and save

In [ ]:
# ── Cell 1: CONFIG ── edit these values before running ───────────────────────

BAM_PATH = r"Z:\pipeline\swift-seq\out_dir\workup\fastqs\100uM\raw\mm10SE\aligned.mouse.sorted.bam"
GTF_PATH = r"Z:\pipeline\scrnaseq-analysis\mm10.ncbiRefSeq.gtf.gz"

# Output paths (relative to this notebook's directory, or absolute)
OUT_H5AD = "bam_count_matrix.h5ad"
OUT_CSV  = "bam_count_matrix.csv"

# MAPQ threshold: 255 = STAR uniquely mapped reads only
MAPQ_THRESHOLD = 255

# Post-counting filters
MIN_CELLS = 1   # exclude genes expressed in fewer than this many cells
MIN_GENES = 1   # exclude cells expressing fewer than this many genes

SHOW_PROGRESS = True  # toggle tqdm progress bar

# Gene name column in the GTF.
# Set to None to auto-detect from: gene_name → gene → gene_id → transcript_id
# If auto-detect fails, set this to the correct column name printed by Cell 4.
GENE_NAME_COL = None

# Number of threads for pysam BAM I/O (increase if you have spare CPU cores)
N_THREADS = 4

# ─────────────────────────────────────────────────────────────────────────────
assert BAM_PATH, "Set BAM_PATH before running."
assert GTF_PATH, "Set GTF_PATH before running."
print("Config OK")

In [ ]:
# Cell 2 — Imports
import re
import collections
import os

import pysam
import pyranges as pr
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print(f"pysam    {pysam.__version__}")
print(f"pyranges {pr.__version__}")
print(f"anndata  {ad.__version__}")

In [ ]:
# Cell 3 — Read-name regex (compiled once for efficiency)
#
# Read name format:
#   AV233703:260221_guttman_others:2532409946:1:10304:1383:1781:TCTGTTCAGT::[NYBot57_Stg][ODD2_D10]-suffix
#
#   Fields 1-7 (colon-separated) : standard Illumina
#   Field 8                      : UMI
#   ::                           : separator
#   [barcode1][barcode2]         : cell barcodes in square brackets
#   -suffix                      : RT primer (ignored)
#
# Cell barcode = barcode1 + "." + barcode2
# Reads where either barcode is NOT_FOUND are discarded.

READ_NAME_RE = re.compile(
    r"^[^:]+:[^:]+:[^:]+:[^:]+:[^:]+:[^:]+:[^:]+:"
    r"([ACGTN]+)"      # group 1 — UMI
    r"::\[([^\]]+)\]"  # group 2 — barcode1
    r"\[([^\]]+)\]"    # group 3 — barcode2
)

NOT_FOUND = "NOT_FOUND"
print("Regex compiled OK")

In [ ]:
# Cell 4 — Load GTF and build interval index
#
# Two structures are built:
#   chrom_ranges : {chrom -> PyRanges}  (kept for the chromosome validation cell)
#   chrom_ncls   : {chrom -> (NCLS, gene_names_array)}
#
# chrom_ncls uses the ncls C library directly, bypassing the Python-level
# pr.PyRanges() object creation that was the main bottleneck (~6 ms/read).
# A direct ncls.find_overlap() call takes ~1-10 µs.

from ncls import NCLS

print(f"Loading GTF: {GTF_PATH}")
gtf_full = pr.read_gtf(GTF_PATH, as_df=False)

print(f"\nAvailable GTF columns: {list(gtf_full.columns)}")

# Use 'gene' rows; fall back to 'transcript' for GTFs that omit gene-level rows
gtf_genes = gtf_full[gtf_full.Feature == "gene"]
if len(gtf_genes) == 0:
    print("  No 'gene' feature rows — falling back to 'transcript' rows")
    gtf_genes = gtf_full[gtf_full.Feature == "transcript"]
print(f"  Using {len(gtf_genes):,} feature rows (Feature='{gtf_genes.Feature.iloc[0]}')")

# ── Resolve gene name column ─────────────────────────────────────────────────
_CANDIDATES = ["gene_name", "gene", "gene_id", "transcript_id"]

if GENE_NAME_COL is not None:
    if GENE_NAME_COL not in gtf_genes.columns:
        raise ValueError(
            f"GENE_NAME_COL='{GENE_NAME_COL}' not found.\n"
            f"Available: {list(gtf_genes.columns)}"
        )
    gene_col = GENE_NAME_COL
    print(f"  Using user-specified gene column: '{gene_col}'")
else:
    gene_col = next((c for c in _CANDIDATES if c in gtf_genes.columns), None)
    if gene_col is None:
        raise ValueError(
            f"Could not find a gene name column. Tried: {_CANDIDATES}\n"
            f"Available: {list(gtf_genes.columns)}\n"
            f"Set GENE_NAME_COL in CONFIG to the correct column name."
        )
    print(f"  Auto-detected gene column: '{gene_col}'")

if gene_col != "gene_name":
    gtf_genes = gtf_genes.assign("gene_name", lambda df: df[gene_col])

# ── Per-chromosome PyRanges shards (for validation cell) ─────────────────────
chrom_ranges: dict = {}
for chrom in gtf_genes.chromosomes:
    chrom_ranges[chrom] = gtf_genes[chrom]

# ── Per-chromosome ncls index (for fast per-read lookup) ─────────────────────
# Each value is (NCLS_object, numpy_array_of_gene_names).
# NCLS.find_overlap(start, end) returns (iv_start, iv_end, row_id) tuples;
# gene_names[row_id] gives the gene symbol in O(1).
print("\nBuilding ncls interval index …")
chrom_ncls: dict = {}
for chrom, pr_shard in chrom_ranges.items():
    df = pr_shard.df
    starts     = df.Start.values.astype(np.int64)
    ends       = df.End.values.astype(np.int64)
    ids        = np.arange(len(df), dtype=np.int64)
    gene_names = df["gene_name"].values          # numpy array, O(1) integer index
    chrom_ncls[chrom] = (NCLS(starts, ends, ids), gene_names)

print(f"Index built for {len(chrom_ncls):,} chromosomes.")
print(f"Example gene names: {list(gtf_genes.gene_name[:5])}")

In [ ]:
# Cell 4b — Validate chromosome name overlap between BAM and GTF
#
# STAR aligns to UCSC-style names (chr1) by default.
# Some GTFs use Ensembl-style (1, 2, ...). A mismatch causes 0% gene assignment.

with pysam.AlignmentFile(BAM_PATH, "rb") as bam:
    bam_chroms = set(h["SN"] for h in bam.header["SQ"])

gtf_chroms = set(chrom_ranges.keys())
overlap    = bam_chroms & gtf_chroms

if len(overlap) == 0:
    print("WARNING: No chromosome name overlap between BAM and GTF!")
    print(f"  BAM examples : {sorted(bam_chroms)[:5]}")
    print(f"  GTF examples : {sorted(gtf_chroms)[:5]}")
    print("  Fix: ensure BAM and GTF use the same naming convention (e.g. both 'chr1' or both '1').")
else:
    print(f"OK: {len(overlap):,} chromosomes match between BAM and GTF.")
    print(f"   BAM total chroms: {len(bam_chroms):,}  |  GTF total chroms: {len(gtf_chroms):,}")

In [ ]:
# Cell 5 — Gene assignment helper (uses ncls directly)

def assign_gene(chrom: str, start: int, end: int, chrom_ncls: dict) -> "str | None":
    """
    Assign a single gene to interval [start, end) on chrom using a direct
    ncls C-level lookup (~1-10 µs) instead of per-read PyRanges objects (~6 ms).

    Returns the gene_name if exactly one gene overlaps, otherwise None.
    Multi-gene overlaps are discarded (standard scRNA-seq convention).
    """
    entry = chrom_ncls.get(chrom)
    if entry is None:
        return None
    ncls_obj, gene_names = entry
    hits = list(ncls_obj.find_overlap(start, end))
    if not hits:
        return None
    unique_genes = {gene_names[h[2]] for h in hits}
    return unique_genes.pop() if len(unique_genes) == 1 else None


print("assign_gene() defined (ncls backend)")

In [ ]:
# Cell 6 — Stream BAM and accumulate per-(cell, gene) UMI sets
#
# Speed optimisations applied:
#   1. assign_gene() uses ncls directly (~1-10 µs) instead of pr.PyRanges() per read (~6 ms)
#   2. STAR tag fast path: if the BAM carries GN (gene name) tags, skip GTF lookup entirely
#   3. threads=N_THREADS for parallel BAM decompression (htslib I/O)

# ── Detect STAR gene tags (check first 200 mapped reads) ─────────────────────
_use_star_tags = False
with pysam.AlignmentFile(BAM_PATH, "rb", threads=N_THREADS) as _bam:
    _checked = 0
    for _r in _bam.fetch(until_eof=True):
        if _r.is_unmapped:
            continue
        if _r.has_tag("GN"):
            _use_star_tags = True
            break
        _checked += 1
        if _checked >= 200:
            break

if _use_star_tags:
    print("STAR GN tags detected — using tag-based gene assignment (fastest path, GTF index unused)")
else:
    print("No STAR GN tags — using ncls GTF index for gene assignment")

# ── Main accumulator ─────────────────────────────────────────────────────────
counts: dict = collections.defaultdict(lambda: collections.defaultdict(set))

n_total        = 0
n_unmapped     = 0
n_low_mapq     = 0
n_no_parse     = 0
n_not_found_bc = 0
n_no_gene      = 0
n_assigned     = 0

print(f"\nStreaming BAM: {BAM_PATH}")

with pysam.AlignmentFile(BAM_PATH, "rb", threads=N_THREADS) as bam:
    for read in tqdm(
        bam.fetch(until_eof=True),
        disable=not SHOW_PROGRESS,
        desc="Processing reads",
        unit=" reads",
        miniters=100_000,
    ):
        n_total += 1

        # Quality filters
        if read.is_unmapped:
            n_unmapped += 1
            continue

        if read.mapping_quality < MAPQ_THRESHOLD:
            n_low_mapq += 1
            continue

        # Parse read name
        m = READ_NAME_RE.match(read.query_name)
        if m is None:
            n_no_parse += 1
            continue

        umi      = m.group(1)
        barcode1 = m.group(2)
        barcode2 = m.group(3)

        if NOT_FOUND in barcode1 or NOT_FOUND in barcode2:
            n_not_found_bc += 1
            continue

        cell_barcode = barcode1 + "." + barcode2

        # Gene assignment — STAR tag fast path, then ncls GTF fallback
        if _use_star_tags:
            if read.has_tag("GN"):
                raw = read.get_tag("GN")
                gene = raw if raw and raw != "-" else None
            else:
                gene = None
        else:
            gene = assign_gene(
                read.reference_name,
                read.reference_start,
                read.reference_end,
                chrom_ncls,
            )

        if gene is None:
            n_no_gene += 1
            continue

        counts[cell_barcode][gene].add(umi)
        n_assigned += 1

print(f"\nDone. {n_total:,} reads processed.")

if n_total > 0 and n_no_parse / n_total > 0.01:
    print(f"WARNING: {n_no_parse/n_total:.1%} of reads did not match the expected read name format.")
    print("  Check that BAM_PATH is correct and the read name format matches Cell 3.")

In [ ]:
# Cell 7 — BAM scan summary

def pct(n, total):
    return f"{100*n/max(total,1):.1f}%"

print("=== Read processing summary ===")
print(f"  Total reads              : {n_total:>12,}")
print(f"  Unmapped    (discarded)  : {n_unmapped:>12,}  ({pct(n_unmapped, n_total)})")
print(f"  Low MAPQ    (discarded)  : {n_low_mapq:>12,}  ({pct(n_low_mapq, n_total)})")
print(f"  Bad read name (skipped)  : {n_no_parse:>12,}  ({pct(n_no_parse, n_total)})")
print(f"  NOT_FOUND barcode        : {n_not_found_bc:>12,}  ({pct(n_not_found_bc, n_total)})")
print(f"  No gene overlap          : {n_no_gene:>12,}  ({pct(n_no_gene, n_total)})")
print(f"  Assigned                 : {n_assigned:>12,}  ({pct(n_assigned, n_total)})")
print()
print(f"Unique cell barcodes : {len(counts):>10,}")
all_genes_set = {g for cd in counts.values() for g in cd}
print(f"Unique genes detected: {len(all_genes_set):>10,}")

In [ ]:
# Cell 8 — Build sorted axis lists and index lookup dicts

all_barcodes = sorted(counts.keys())
all_genes    = sorted({g for cd in counts.values() for g in cd})

n_cells = len(all_barcodes)
n_genes = len(all_genes)

print(f"Matrix dimensions: {n_cells:,} cells × {n_genes:,} genes")

barcode_idx = {bc: i for i, bc in enumerate(all_barcodes)}
gene_idx    = {g:  j for j, g  in enumerate(all_genes)}

In [ ]:
# Cell 9 — Build sparse count matrix
#
# scipy.sparse.lil_matrix supports efficient random element assignment.
# Converted to CSR (required by AnnData) after filling.

print("Filling sparse matrix …")
mat = sp.lil_matrix((n_cells, n_genes), dtype=np.int32)

for cell_bc, gene_dict in tqdm(counts.items(), desc="Cells", disable=not SHOW_PROGRESS):
    row = barcode_idx[cell_bc]
    for gene_name, umi_set in gene_dict.items():
        col = gene_idx[gene_name]
        mat[row, col] = len(umi_set)  # UMI deduplication: set cardinality

mat_csr = mat.tocsr()
print(f"Matrix density: {mat_csr.nnz / max(n_cells * n_genes, 1):.4%}")
print(f"Total UMIs    : {int(mat_csr.sum()):,}")

In [ ]:
# Cell 10 — Construct AnnData and apply cell/gene filters

adata = ad.AnnData(
    X   = mat_csr,
    obs = pd.DataFrame(index=all_barcodes),
    var = pd.DataFrame(index=all_genes),
)
adata.obs_names.name = "cell_barcode"
adata.var_names.name = "gene_name"

n_cells_raw, n_genes_raw = adata.shape

# Gene filter: expressed in at least MIN_CELLS cells
genes_per_cell = np.array((adata.X > 0).sum(axis=0)).flatten()
gene_mask      = genes_per_cell >= MIN_CELLS

# Cell filter: expresses at least MIN_GENES genes
cells_per_gene = np.array((adata.X > 0).sum(axis=1)).flatten()
cell_mask      = cells_per_gene >= MIN_GENES

adata = adata[cell_mask, :][:, gene_mask].copy()

print(f"Before filtering : {n_cells_raw:,} cells × {n_genes_raw:,} genes")
print(f"After  filtering : {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"  (removed {n_cells_raw - adata.n_obs:,} cells, {n_genes_raw - adata.n_vars:,} genes)")

In [ ]:
# Cell 11 — UMI distribution QC plots

if sp.issparse(adata.X):
    total_counts = np.array(adata.X.sum(axis=1)).flatten()
else:
    total_counts = adata.X.sum(axis=1).flatten()

adata.obs["total_counts"] = total_counts
adata.obs["n_genes"]      = np.array((adata.X > 0).sum(axis=1)).flatten()

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
pct_values  = np.percentile(total_counts, percentiles)
pct_df = pd.DataFrame({"Percentile": percentiles, "UMI count": pct_values.astype(int)})
print(pct_df.to_string(index=False))
print(f"\nMin: {total_counts.min():.0f}  |  Max: {total_counts.max():.0f}  |  Mean: {total_counts.mean():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(total_counts, bins=100, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Total UMI count per cell")
axes[0].set_ylabel("Number of cells")
axes[0].set_title("UMI count distribution")

axes[1].hist(np.log10(total_counts + 1), bins=100, color="steelblue", edgecolor="none")
axes[1].set_xlabel("log10(Total UMI count + 1)")
axes[1].set_ylabel("Number of cells")
axes[1].set_title("UMI count distribution (log10)")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — Save outputs

nb_dir = os.path.dirname(os.path.abspath("__file__"))

h5ad_path = OUT_H5AD if os.path.isabs(OUT_H5AD) else os.path.join(nb_dir, OUT_H5AD)
csv_path  = OUT_CSV  if os.path.isabs(OUT_CSV)  else os.path.join(nb_dir, OUT_CSV)

# Save AnnData
adata.write_h5ad(h5ad_path)
print(f"Saved AnnData : {h5ad_path}")

# Save CSV count matrix (sparse-aware to avoid dense materialisation)
counts_df = pd.DataFrame.sparse.from_spmatrix(
    adata.X,
    index=adata.obs_names,
    columns=adata.var_names,
)
counts_df.to_csv(csv_path)
print(f"Saved CSV     : {csv_path}")

print(f"\nFinal matrix  : {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"Total UMIs    : {int(adata.X.sum()):,}")
print("\nTo continue analysis, load the .h5ad in gene_expression_analysis.ipynb:")
print(f'  H5AD_PATH = r"{h5ad_path}"')